# PIBIC · Treino organizado da expansão de 1 GDL

Este notebook executa o comando publicado do experimento `expansao_1gdl_v1`. Baixa código, configuração e CSV da revisão identificada e confere os três hashes. O banco tem 2.520 simulações, agregadas em 1.080 condições, com avaliação por famílias em cinco rodadas e quatro indicadores de saída.

Execute as células em ordem. A documentação metodológica e as evidências estão no [repositório](https://github.com/tedescohf19/pibic-suspensao-veicular). Os notebooks anteriores estão preservados em `notebooks/historico/`.


In [ ]:
# Carregar a revisão e conferir a integridade
import hashlib
import json
import sys
import subprocess
from datetime import datetime
from pathlib import Path
from urllib.request import urlopen

REVISAO = '3bb3f2d27c3929b66afb47397d766ede630b45dd'
ARQUIVOS = {'src/1gdl/mlp/treinar_expansao_1gdl.py': 'b515002fdbd10171dfc543e8447d68dd7cc285803002402846869ef387e3c15d', 'experimentos/expansao_1gdl_v1.json': '41b5eda526304addaf94197af4e0eee4fa92b27faf765f256c97358dfedd9e28', 'dados/1gdl/expansao/banco_1gdl_expandido_2520.csv': '43b95bdb2feea3de9835051884980797f3289db23cea4bb68470b33500e65798'}
RAIZ = Path('/content/pibic-expansao-1gdl')
for nome, esperado in ARQUIVOS.items():
    url = f'https://raw.githubusercontent.com/tedescohf19/pibic-suspensao-veicular/{REVISAO}/{nome}'
    with urlopen(url, timeout=60) as resposta:
        conteudo = resposta.read()
    if hashlib.sha256(conteudo).hexdigest() != esperado:
        raise ValueError(f'Conteúdo divergente na revisão: {nome}')
    destino = RAIZ / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(conteudo)
print('Código, configuração e banco conferidos. Revisão', REVISAO)


In [ ]:
# Executar as cinco rodadas com o comando do repositório
import os

SAIDA = RAIZ / 'gerados' / ('colab_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
ambiente = dict(os.environ, MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
subprocess.run([sys.executable, str(RAIZ / 'src/1gdl/mlp/treinar_expansao_1gdl.py'),
                '--saida', str(SAIDA)], check=True, env=ambiente)


In [ ]:
# Mostrar relatório, métricas e gráficos da execução
import pandas as pd
from IPython.display import Markdown, Image, display

display(Markdown((SAIDA / 'relatorio_mlp_1gdl.md').read_text(encoding='utf-8')))
display(pd.read_csv(SAIDA / 'metricas_por_categoria.csv'))
display(pd.read_csv(SAIDA / 'particoes_por_familia.csv'))
for nome in ['curvas_aprendizagem.png', 'paridade_teste.png', 'erros_por_categoria.png']:
    display(Image(filename=str(SAIDA / nome)))
print('Pesos, normalizadores, tabelas e proveniência em', SAIDA)


## Guardar os resultados

Baixe o ZIP antes de encerrar a sessão. Ele contém os cinco checkpoints, as tabelas, gráficos, configuração e proveniência da execução. Os modelos ficam associados às suas famílias de treino, validação e teste.


In [ ]:
# Baixar o pacote completo da execução
from google.colab import files

files.download(str(SAIDA) + '.zip')
